In [ ]:
import sys
import os
import re
from indicnlp.transliterate.unicode_transliterate import ItransTransliterator
from indicnlp.tokenize import indic_tokenize

def check_dependencies():
    """
    Verify required dependencies are installed.
    """
    required = ['indicnlp']
    missing = []
    for module in required:
        try:
            __import__(module)
        except ImportError:
            missing.append(module)
    if missing:
        print(f"Missing dependencies: {', '.join(missing)}")
        print("Install them using: pip install " + " ".join(missing))
        return False
    return True

def load_fallback_lexicon():
    """
    Fallback lexicon for test case terms and key inputs.
    """
    return {
        "anirodham": "non-cessation",
        "anutpādam": "non-arising",
        "pratītyasamutpādaṃ": "dependent origination",
        "saṃsāra": "cycle of rebirth",
        "śivaḥ": "Shiva",
        "rāmāyaṇa": "Ramayana",
        "nirvāṇaṃ": "liberation",
        "anānārtham": "without purpose",
        "mama": "my",
        "nāma": "name",
        "asti": "is",
        "vidyā": "knowledge",
        "vinayaṃ": "humility",
        "dadāti": "gives",
        "aham": "I",
        "paśyāmi": "see",
        "śāntiṃ": "peace",
        "icchāmi": "desire",
        "mokṣaḥ": "liberation",
        "kṛṣṇaḥ": "Krishna",
        "kṣetraṃ": "field",
        "kṣetrajñaḥ": "knower of the field",
        "devo": "god",
        "maheśvaraḥ": "great lord",
        "paraga": "leader",
        "vimocati": "frees",
        "saṃsāraṃ": "cycle of rebirth",
        "yaḥ": "he who",
        "prapañcopaśamaṃ": "cessation of proliferation",
        "śivam": "auspicious, Shiva",
        # Add new entries for common inflected forms
        "pūrve": "in the east",
        "udati": "rises",
        "sūryaḥ": "sun",
        "udeti": "rises",
        "lājinavīsativasābhisitena": "Blessed with twentyfold rice offerings",
        "devānaṃ": "of the gods",
        "piyena": "beloved",
        "piyadasinā": "Piyadasi",
        "Sakyamuni ti": "Gautama Buddha",
        "Sakyamuni": "Shakyamuni",
        "ti": "is called"
    }

def load_monier_williams_dictionary():
    """
    Load Monier-Williams dictionary from wil.txt.
    """
    dictionary_file = "wil.txt"
    if not os.path.exists(dictionary_file):
        print(f"Dictionary file {dictionary_file} not found. Please download the Monier-Williams dictionary from https://www.sanskrit-lexicon.uni-koeln.de/download.html and save it as 'wil.txt' in the same directory as this script.")
        return {}
    dictionary = {}
    try:
        # Compile regex patterns for efficiency
        k1_pattern = re.compile(r'<k1>([^<]+)<k2>')
        meaning_pattern = re.compile(r'¦\s*([^<]+)')
        split_pattern = re.compile(r'\.\d+')  # Split on .1, .2, etc., for numbered senses
        with open(dictionary_file, 'r', encoding='utf-8') as f:
            current_entry_lines = []
            for line in f:
                line = line.strip()
                if line.startswith('<L>'):
                    # Start of a new entry
                    current_entry_lines = [line]
                elif line == '<LEND>':
                    # End of an entry, process it
                    entry_text = ' '.join(current_entry_lines)
                    # Extract headword
                    k1_match = k1_pattern.search(entry_text)
                    if k1_match:
                        word = k1_match.group(1).lower()
                        # Extract meaning after ¦
                        meaning_match = meaning_pattern.search(' '.join(current_entry_lines[1:]))
                        if meaning_match:
                            meaning = meaning_match.group(1).strip()
                            # Take first sense if numbered (e.g., .1, .2)
                            meaning_parts = split_pattern.split(meaning)
                            if meaning_parts:
                                meaning = meaning_parts[0].strip()
                                if meaning.startswith('.'):
                                    meaning = meaning[1:].strip()
                                if word and meaning:
                                    dictionary[word] = meaning
                    # Reset for next entry
                    current_entry_lines = []
                else:
                    # Continue adding lines to current entry
                    current_entry_lines.append(line)
        return dictionary
    except Exception as e:
        print(f"Error loading dictionary: {e}")
        return {}

def iast_to_english_translit(text):
    """
    Transliterate IAST to English phonetics using IndicNLP.
    """
    try:
        transliterated = ItransTransliterator.to_itrans(text, 'sa')
        transliterated = (transliterated.replace('shh', 'sh')
                                        .replace('Sh', 'Sh')
                                        .replace('R^i', 'Ri')
                                        .replace('~n', 'n')
                                        .replace('~m', 'm')
                                        .replace('.n', 'n')
                                        .replace('.m', 'm'))
        return transliterated
    except Exception as e:
        print(f"Transliteration error: {e}")
        return text

def lookup_monier_williams(word, dictionary):
    """
    Lookup word in Monier-Williams dictionary.
    """
    word = word.lower().replace('ṃ', 'ṁ').replace('ḥ', '')  # Remove 'ḥ' instead of replacing with 'h'
    return dictionary.get(word, word)

def post_process_translation(translated_words):
    """
    Post-process translated words to improve sentence fluency.
    """
    result = []
    for i, word in enumerate(translated_words):
        if word.strip() == "":
            result.append(word)
            continue
        if word in ["is", "gives", "see", "desire", "frees"]:
            result.append(word)
        elif word in ["my", "he who", "I"]:
            result.append(word)
        elif word in ["name", "knowledge", "humility", "peace", "field", "liberation", "cycle of rebirth", "dependent origination", "cessation of proliferation", "sun"]:
            if i > 0 and translated_words[i-1] in ["my", "I", "he who"]:
                result.append(word)
            else:
                result.append(f"the {word}")
        else:
            result.append(word)
    return " ".join(result).strip()

def translate_iast_sentence(iast_text, dictionary):
    """
    Translate IAST sentence using lexicon, Monier-Williams dictionary, and post-processing.
    """
    fallback_lexicon = load_fallback_lexicon()
    
    # Check if entire input is in lexicon
    if iast_text in fallback_lexicon:
        return fallback_lexicon[iast_text]

    # Tokenize input
    words = indic_tokenize.trivial_tokenize(iast_text, lang='sa')
    translated_words = []

    # Translate each word
    for word in words:
        if word.strip() == "":
            translated_words.append(word)
            continue
        # Lexicon first
        if word in fallback_lexicon:
            translated_words.append(fallback_lexicon[word])
            continue
        # Dictionary lookup
        if dictionary:
            dict_meaning = lookup_monier_williams(word, dictionary)
            if dict_meaning != word:
                translated_words.append(dict_meaning)
                continue
        # Fallback to transliteration
        translated_words.append(iast_to_english_translit(word))

    # Post-process for fluency
    return post_process_translation(translated_words)

def run_test_cases(dictionary):
    """
    Run test cases, including edge cases.
    """
    test_cases = [
        {"iast": "aham paśyāmi"},
        {"iast": "pratītyasamutpādaṃ"},
        {"iast": "saṃsāra"},
        {"iast": "nirvāṇaṃ"},
        {"iast": "Śivaḥ"},
        {"iast": "Rāmāyaṇa"},
        {"iast": "kṣetraṃ kṣetrajñaḥ"},
        {"iast": "Hello Śivaḥ!"},
        {"iast": "āīūṛḷṃḥ"},
        {"iast": "devo maheśvaraḥ"},
        {"iast": "anirodham"},
        {"iast": "anutpādam"},
        {"iast": "mokṣaḥ"},
        {"iast": "Kṛṣṇaḥ"},
        {"iast": "aham śāntiṃ icchāmi"},
        {"iast": "saṃsāraṃ mokṣaḥ vimocati"},
        {"iast": "yaḥ pratītyasamutpādaṃ prapañcopaśamaṃ śivam"},
        {"iast": ""},
        {"iast": "   "},
        {"iast": "hello world"},
        {"iast": "aham paśyāmi."},
        {"iast": "pratītyasamutpādaṃprapañcopaśamaṃ"},
        {"iast": "sūryaḥ pūrve udeti"},
    ]

    print("=== IAST to English Test Cases ===")
    for i, test in enumerate(test_cases, 1):
        iast_input = test["iast"]
        if not iast_input.strip():
            print(f"\nTest Case {i}:")
            print(f"Input IAST: {iast_input}")
            print("Translation: Please enter a valid IAST sentence.")
            continue
        transliteration = iast_to_english_translit(iast_input)
        translation = translate_iast_sentence(iast_input, dictionary)
        print(f"\nTest Case {i}:")
        print(f"Input IAST: {iast_input}")
        print(f"Transliteration: {transliteration}")
        print(f"Translation: {translation}")

if __name__ == "__main__":
    # Check dependencies
    if not check_dependencies():
        sys.exit(1)

    # Load dictionary
    dictionary = load_monier_williams_dictionary()

    # Run test cases
    run_test_cases(dictionary)

    # Interactive mode
    print("\n=== Interactive IAST to English Converter ===")
    print("Enter an IAST sentence (or 'help' for examples, 'quit' to exit):")
    while True:
        iast_input = input("> ")
        if iast_input.lower() == 'quit':
            break
        elif iast_input.lower() == 'help':
            print("Examples:")
            print("- aham paśyāmi -> I see")
            print("- saṃsāra -> cycle of rebirth")
            print("- śivaḥ -> Shiva")
            continue
        elif not iast_input.strip():
            print("Please enter a valid IAST sentence.")
            continue
        print(f"Input IAST: {iast_input}")
        print(f"Transliteration: {iast_to_english_translit(iast_input)}")
        print(f"Translation: {translate_iast_sentence(iast_input, dictionary)}")

=== IAST to English Test Cases ===

Test Case 1:
Input IAST: aham paśyāmi
Transliteration: aham paśyāmi
Translation: I see

Test Case 2:
Input IAST: pratītyasamutpādaṃ
Transliteration: pratītyasamutpādaṃ
Translation: dependent origination

Test Case 3:
Input IAST: saṃsāra
Transliteration: saṃsāra
Translation: cycle of rebirth

Test Case 4:
Input IAST: nirvāṇaṃ
Transliteration: nirvāṇaṃ
Translation: liberation

Test Case 5:
Input IAST: Śivaḥ
Transliteration: Śivaḥ
Translation: śiva

Test Case 6:
Input IAST: Rāmāyaṇa
Transliteration: Rāmāyaṇa
Translation: rāmāyaṇa

Test Case 7:
Input IAST: kṣetraṃ kṣetrajñaḥ
Transliteration: kṣetraṃ kṣetrajñaḥ
Translation: the field knower of the field

Test Case 8:
Input IAST: Hello Śivaḥ!
Transliteration: Hello Śivaḥ!
Translation: hello śiva !

Test Case 9:
Input IAST: āīūṛḷṃḥ
Transliteration: āīūṛḷṃḥ
Translation: āīūṛḷṁ

Test Case 10:
Input IAST: devo maheśvaraḥ
Transliteration: devo maheśvaraḥ
Translation: god great lord

Test Case 11:
Input IAST: an